# Walkthrough — what the eigenvalues *mean*, on one 400-track event

A from-scratch, picture-led explanation. We use one fixed-ε ($`\varepsilon=2`$ mrad,
$`\gamma=3,\delta=1`$) 400-track event, solved classically and with the 1BQF.

## Terminology (read this first)
| Term | Meaning |
|---|---|
| **Hit** | A space-point where a particle crossed one of the **5 detector planes** (at $`z=33,66,99,132,165`$ mm). Carries $`(x,y,z)`$ and a truth `track_id`. |
| **Track** | The true trajectory of one particle = its 5 hits (one per plane). "400 tracks" = 400 particles. |
| **Segment** | A *candidate* straight link between a hit on one plane and a hit on the **next** plane. The solver must switch each segment ON or OFF. Its value $`x_i\in\mathbb{R}`$ is the **activation**. |
| **True segment** | Both its hits belong to the **same** track — a genuine piece of a trajectory. Each track has **4** (it spans 4 plane-gaps). |
| **False segment** | Its two hits belong to **different** tracks — a spurious link. These are the overwhelming majority. |
| **Matrix $`A`$** | Encodes "segments that could be the same particle reinforce each other": $`A_{ii}=\gamma+\delta`$, and $`A_{ij}=-1`$ if segments $`i,j`$ **share a middle hit** and their kink angle is $`<\varepsilon`$ (call them *compatible*). Solving $`A\mathbf{x}=\delta\mathbf{1}`$ gives the activations. |
| **Compatible / coupling** | Two segments are coupled iff they share a hit and are angle-aligned. The off-diagonal of $`A`$. |
| **Cluster** | A connected group of mutually-coupled segments (a connected component of the coupling graph). An **isolated** segment is a cluster of size 1; a **real track** is a cluster of 4 (its chain); a **bridge** is an accidental cluster of false segments. |
| **Eigenvalue $`\lambda`$ / eigenvector $`\mathbf{u}`$** | $`A\mathbf{u}=\lambda\mathbf{u}`$. $`A`$ is **block-diagonal over clusters**, so each cluster has its own little set of eigenvalues. An isolated segment is its *own* eigenvector with eigenvalue **exactly** $`\gamma+\delta`$. |
| **1BQF (quantum solver)** | Inverts $`A`$ one eigenvalue at a time, but with **one bit** of precision the inversion is the **filter** $`f(\lambda)=\cos(\lambda t/2)`$, $`t=\pi/(\gamma+\delta)`$ — instead of the exact classical $`1/\lambda`$. |
| **"Bad" eigenvalue (the notch)** | $`\lambda=\gamma+\delta`$, where $`f(\gamma+\delta)=\cos\frac\pi2=0`$: that eigenvector's amplitude is **erased**. |
| **"Good" eigenvalue** | Any $`\lambda`$ away from the notch, where $`f(\lambda)\ne0`$: that eigenvector is **kept**. |
| **Threshold $`\tau`$** | A segment is declared active iff $`x_i>\tau=\delta/(\delta+\gamma)+0.10=0.35`$. |

## 0b. The whole story in one schematic
The real event is far too dense to see this (segments overlap to the pixel), so
here is a **schematic** — three well-separated, idealised clusters, one of each
type, with what the 1-bit quantum filter does to each. Green solid = a **true**
segment; red dashed = a **false** segment. Planes are drawn as translucent sheets.

In [1]:
import sys
sys.path.insert(0, "/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Segment_level_studies")
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib.lines import Line2D
import seg_store as S
_OUT = Path(S.__file__).resolve().parent / "outputs" / "walkthrough_400"; _OUT.mkdir(parents=True, exist_ok=True)
GREEN, RED = "#1b7837", "#d6604d"
T1, T2 = "#6a3d9a", "#ff7f00"   # two distinct track colours for the isolated-false endpoints

fig = plt.figure(figsize=(15, 10)); ax = fig.add_subplot(111, projection="3d")
zp = np.arange(5.0)                                   # 5 detector planes
for z in zp:
    v = [[(-3.3, -9.5, z), (3.3, -9.5, z), (3.3, 9.2, z), (-3.3, 9.2, z)]]
    ax.add_collection3d(Poly3DCollection(v, alpha=0.045, facecolor="steelblue", edgecolor="grey", lw=0.4))

def chain(xs, ys, col):                               # a true 4-segment track
    for k in range(4):
        ax.plot([xs[k], xs[k+1]], [ys[k], ys[k+1]], [zp[k], zp[k+1]], color=col, lw=3, zorder=5)
    ax.scatter(xs, ys, zp, color=col, s=45, ec="k", lw=0.5, zorder=6)

# (A) REAL TRACK  — top band, all true (kept)
chain(np.linspace(-1.2, 1.2, 5), 6.4 + np.linspace(0, 0.9, 5), GREEN)
ax.text(0.0, 6.8, 4.7, "A", color=GREEN, fontsize=20, fontweight="bold", ha="center")
# (B) ISOLATED FALSE  — middle band, one link between two *different* tracks' hits, no neighbour
hA = (-1.5, 0.7, zp[1]); hB = (1.2, -0.3, zp[2])
ax.scatter([hA[0]], [hA[1]], [hA[2]], color=T1, s=60, ec="k", lw=0.6, zorder=6)
ax.scatter([hB[0]], [hB[1]], [hB[2]], color=T2, s=60, ec="k", lw=0.6, zorder=6)
ax.plot([hA[0], hB[0]], [hA[1], hB[1]], [hA[2], hB[2]], color=RED, ls="--", lw=2.6, zorder=5)
ax.text(-0.2, 0.2, 3.0, "B", color=RED, fontsize=20, fontweight="bold", ha="center")
# (C) CROSS-TRACK BRIDGE — bottom band, two ~parallel tracks + red cross-links (false, kept)
b1x = np.linspace(-1.1, 1.1, 5); b1y = np.full(5, -6.1)
b2x = np.linspace(-0.5, 1.7, 5); b2y = np.full(5, -7.1)
chain(b1x, b1y, GREEN); chain(b2x, b2y, GREEN)
ax.plot([b1x[1], b2x[2]], [b1y[1], b2y[2]], [zp[1], zp[2]], color=RED, ls="--", lw=2.6, zorder=8)
ax.plot([b2x[2], b1x[3]], [b2y[2], b1y[3]], [zp[2], zp[3]], color=RED, ls="--", lw=2.6, zorder=8)
ax.text(0.6, -6.6, 4.7, "C", color=RED, fontsize=20, fontweight="bold", ha="center")

ax.set_xlabel("x"); ax.set_ylabel("y (clusters drawn well-separated)"); ax.set_zlabel("detector plane (z) →")
ax.set_yticks([]); ax.set_zticks(range(5)); ax.set_zticklabels([f"p{i+1}" for i in range(5)])
ax.set_box_aspect((0.9, 2.1, 1.0)); ax.view_init(elev=20, azim=-68)
handles = [Line2D([0], [0], color=GREEN, lw=3, label="true segment (kept)"),
           Line2D([0], [0], color=RED, lw=2.6, ls="--", label="false segment")]
ax.legend(handles=handles, loc="upper left", fontsize=10, framealpha=0.95)
fig.suptitle("Schematic: the three segment types and what the 1-bit quantum filter does to each",
             fontsize=14, fontweight="bold", y=0.94)
key = ("A   real track (4-segment chain):  eigenvalues {2.4, 3.4, 4.6, 5.6} are OFF the notch   ->   quantum KEEPS it  (correct; keeps 3/4)\n"
       "B   isolated false segment (no compatible neighbour):  eigenvalue = gamma+delta = 4 = the NOTCH   ->   quantum ERASES it  (correct)\n"
       "C   cross-track bridge (two near-parallel tracks accidentally aligned):  eigenvalues OFF the notch   ->   quantum KEEPS it  (FALSE POSITIVE)")
fig.text(0.5, 0.07, key, ha="center", va="top", fontsize=10, family="monospace",
         bbox=dict(boxstyle="round", fc="#fbfbf0", ec="grey", alpha=0.95))
fig.subplots_adjust(bottom=0.16, top=0.92)
for e, dp in (("pdf", 600), ("png", 300)):
    fig.savefig(_OUT / f"schematic_3d_types.{e}", dpi=dp, bbox_inches="tight", facecolor="white")
plt.show(); print("saved schematic_3d_types")

saved schematic_3d_types


In [2]:
import sys
sys.path.insert(0, "/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Segment_level_studies")
from pathlib import Path
import numpy as np, scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt
import seg_store as S, qtrk_pipeline as qp

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
OUT = Path(S.__file__).resolve().parent / "outputs" / "walkthrough_400"
OUT.mkdir(parents=True, exist_ok=True)
G, D = 3.0, 1.0; SDIAG = G + D; TVAL = np.pi / SDIAG; TAU = S.threshold(G)
f_filter = lambda lam: np.cos(lam * TVAL / 2)            # the 1-bit inversion filter

CI = S.solves_index("classical", gamma=G, hit_ineff=0.0)
QI = S.solves_index("quantum",   gamma=G, hit_ineff=0.0)
q = QI[QI.n_trk == 400].iloc[0]
c = CI[(CI.event_key == q.event_key) & (CI.ham_key == q.ham_key)].iloc[0]
solC = np.asarray(qp.load_solution(c.sol_key)["sol"], float)
solQ = qp.rescale_to_signal(np.asarray(qp.load_solution(q.sol_key)["sol"], float), solC, TAU)
ev = S._event_of(c); ham = qp.build_hamiltonian(ev, epsilon=float(c.epsilon), gamma=G, delta=D)
A = ham.A.tocsr(); n = ham.n_segments
truth = np.asarray(qp.truth_from_event(ev), bool)
tid = np.asarray(ham._segment_track_ids); xyz = np.asarray(ham._segment_endpoints_xyz)
Cm = (SDIAG*sp.identity(n, format="csr") - A); Cm.setdiag(0); Cm.eliminate_zeros(); Cm = (abs(Cm) > 1e-9).astype(np.int8)
deg = np.asarray(Cm.sum(1)).ravel(); ncomp, lab = connected_components(Cm, directed=False); csize = np.bincount(lab)
print(f"event: {n} candidate segments, {int(truth.sum())} true (= 4 x 400), "
      f"{n-int(truth.sum())} false; tau={TAU}, notch lambda={SDIAG}")
def block(members):                # the little A-block of a cluster + its eigenvalues
    sub = A[members][:, members].toarray(); return sub, np.linalg.eigvalsh(sub)

event: 640000 candidate segments, 1600 true (= 4 x 400), 638400 false; tau=0.35, notch lambda=4.0


## 1. The event, and what a "segment" is
Left: the whole 400-track event — every faint line is a **true** segment, so each
4-segment chain from the beam region is one particle's track. Right: a zoom onto
two neighbouring planes showing that a **segment** is just a hit-to-next-plane link;
the solver's job is to keep the on-track ones (true) and reject the cross-links
(false).

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5.2))
# (a) whole event: all true segments as faint lines (z vs x)
tm = np.where(truth)[0]
for i in tm:
    p0, p1 = xyz[i, 0], xyz[i, 1]
    ax[0].plot([p0[2], p1[2]], [p0[0], p1[0]], "-", color="#1f78b4", lw=0.4, alpha=0.35)
ax[0].set_xlabel("z (mm)"); ax[0].set_ylabel("x (mm)")
ax[0].set_title("(a) The 400-track event\n(each faint line = one true segment)", fontweight="bold")

# (b) zoom: hits of a few tracks on planes 0-1, with true (solid) and false (dashed) candidate links
zlo = 33; planes = [33, 66]
# pick segments between plane0->1 within a small x-window, a handful of tracks
band = [i for i in range(n) if abs(xyz[i,0,2]-33) < 1 and -8 < xyz[i,0,0] < 8 and abs(xyz[i,1,2]-66) < 1]
band = band[:400]
for i in band:
    p0, p1 = xyz[i, 0], xyz[i, 1]
    isT = truth[i]
    ax[1].plot([p0[2], p1[2]], [p0[0], p1[0]], "-" if isT else "--",
               color="#1b7837" if isT else "#d6604d", lw=2 if isT else 0.8, alpha=0.9 if isT else 0.5)
pts = np.array([[xyz[i,e,2], xyz[i,e,0]] for i in band for e in (0,1)])
if len(pts): ax[1].scatter(pts[:,0], pts[:,1], s=30, color="k", zorder=5)
ax[1].set_xlabel("z (mm)"); ax[1].set_ylabel("x (mm)")
ax[1].set_title("(b) Zoom: hits on two planes;\nsegment = a hit-to-next-plane link (green=true, red dashed=false)",
                fontweight="bold")
fig.tight_layout()
for e, dp in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"event_and_segments.{e}", dpi=dp, bbox_inches="tight", facecolor="white")
plt.show(); print("saved event_and_segments")

saved event_and_segments


## 2. Three worked examples — where the eigenvalues come from
$`A`$ splits into independent **clusters**. We take three real clusters from this
event and look at each one's little $`A`$-block, its eigenvalues, and what the
classical ($`1/\lambda`$) and quantum ($`\cos(\lambda t/2)`$) inversions do to it.

- **(A) An isolated false segment** — a cluster of size 1.
- **(B) A real track** — a clean 4-segment chain.
- **(C) A cross-track bridge** — an accidental cluster of false segments.

In [4]:
# ---- select the three clusters (as in the analysis) ----
# (A) isolated false segment
A_iso = np.where((deg == 0) & (~truth))[0][0]
# (B) clean true track: a size-4 component, all 4 true, single track
B_track = None
for ci in np.where(csize == 4)[0]:
    m = np.where(lab == ci)[0]
    tk = set(int(t_) for i in m for t_ in tid[i])
    if truth[m].all() and len(tk) == 1:
        B_track = m; break
# (C) cross-track bridge: a size-3 component hosting a false-active segment
fa = np.where((~truth) & (solQ > TAU))[0]
C_bridge = np.where(lab == lab[[i for i in fa if csize[lab[i]] == 3][0]])[0]

for name, mem in [("(A) isolated false segment", [A_iso]), ("(B) real track (4-chain)", B_track),
                  ("(C) cross-track bridge", C_bridge)]:
    sub, eig = block(mem)
    print(f"\n{name}: {len(mem)} segment(s)")
    print("  A-block =\n", sub.astype(int) if sub.size <= 36 else f"{sub.shape} matrix")
    print("  eigenvalues lambda      :", np.round(eig, 3))
    print("  filter f(lambda)=cos(lt/2):", np.round(f_filter(eig), 3),
          " <- 0 means erased (notch); nonzero means kept")
    print("  classical x :", np.round(solC[mem], 3))
    print("  quantum   x :", np.round(solQ[mem], 3), " (active if > tau=0.35)")
    print("  segment track-id pairs:", [tuple(map(int, tid[i])) for i in mem])


(A) isolated false segment: 1 segment(s)
  A-block =
 [[4]]
  eigenvalues lambda      : [4.]
  filter f(lambda)=cos(lt/2): [0.]  <- 0 means erased (notch); nonzero means kept
  classical x : [0.25]
  quantum   x : [0.]  (active if > tau=0.35)
  segment track-id pairs: [(0, 1)]

(B) real track (4-chain): 4 segment(s)
  A-block =
 [[ 4 -1  0  0]
 [-1  4 -1  0]
 [ 0 -1  4 -1]
 [ 0  0 -1  4]]
  eigenvalues lambda      : [2.382 3.382 4.618 5.618]
  filter f(lambda)=cos(lt/2): [ 0.593  0.24  -0.24  -0.593]  <- 0 means erased (notch); nonzero means kept
  classical x : [0.364 0.455 0.455 0.364]
  quantum   x : [0.361 0.567 0.454 0.182]  (active if > tau=0.35)
  segment track-id pairs: [(0, 0), (0, 0), (0, 0), (0, 0)]

(C) cross-track bridge: 3 segment(s)
  A-block =
 [[ 4  0 -1]
 [ 0  4 -1]
 [-1 -1  4]]
  eigenvalues lambda      : [2.586 4.    5.414]
  filter f(lambda)=cos(lt/2): [ 0.527  0.    -0.527]  <- 0 means erased (notch); nonzero means kept
  classical x : [0.357 0.357 0.429]
  quant

In [5]:
# ---- draw the three clusters in (z, x), hits coloured by track ----
import matplotlib.cm as cm
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
def draw_cluster(a, members, title, note):
    tracks = sorted(set(int(t_) for i in members for t_ in tid[i] if t_ >= 0))
    col = {tk: cm.tab10(k % 10) for k, tk in enumerate(tracks)}
    for i in members:
        p0, p1 = xyz[i, 0], xyz[i, 1]
        isT = (tid[i, 0] == tid[i, 1]) and tid[i, 0] >= 0
        a.plot([p0[2], p1[2]], [p0[0], p1[0]], "-" if isT else "--",
               color="#1b7837" if isT else "#d6604d", lw=3 if isT else 1.8)
        for e2 in (0, 1):
            a.scatter(xyz[i, e2, 2], xyz[i, e2, 0], s=90, zorder=5,
                      color=col.get(int(tid[i, e2]), "0.5"), ec="k", lw=0.6)
    a.set_xlabel("z (mm)"); a.set_ylabel("x (mm)")
    a.set_title(title, fontweight="bold", fontsize=10)
    a.text(0.5, -0.22, note, transform=a.transAxes, ha="center", va="top", fontsize=8.5)

_, eA = block([A_iso]); _, eB = block(B_track); _, eC = block(C_bridge)
draw_cluster(ax[0], [A_iso], "(A) isolated FALSE segment",
             f"1 cluster, no neighbours -> eigenvalue lambda={SDIAG:.0f} (NOTCH/bad)\n"
             f"classical x=0.25, quantum x~0  -> correctly ERASED")
draw_cluster(ax[1], B_track, "(B) a REAL track (4-segment chain)",
             f"eigenvalues {np.round(eB,2)} (all off-notch = GOOD)\n"
             f"quantum keeps 3/4 (one outer drops to 0.18) -> track survives")
draw_cluster(ax[2], C_bridge, "(C) cross-track BRIDGE (false)",
             f"eigenvalues {np.round(eC,2)} (2 good + 1 on notch)\n"
             f"hits from {sorted(set(int(t_) for i in C_bridge for t_ in tid[i]))} -> FALSE POSITIVE survives")
fig.suptitle("Same rule, three clusters: dots = hits (colour = track), green=true segment, red dashed=false segment",
             fontweight="bold", y=1.02)
fig.tight_layout()
for e, dp in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"three_examples.{e}", dpi=dp, bbox_inches="tight", facecolor="white")
plt.show(); print("saved three_examples")

saved three_examples


## 3. The one picture that ties it together — the filter and the notch
The quantum solver keeps each eigenvector in proportion to $`f(\lambda)=\cos(\lambda t/2)`$.
Mark our three examples' eigenvalues on it: the isolated false segment sits **on
the notch** (erased — good, it is false); the real track's four eigenvalues are
**off the notch** (kept — good, it is real); the bridge's eigenvalues are **also
off the notch** (kept — but it is false). The filter cannot tell a real 4-chain
from an accidental cross-track chain: both have "good" eigenvalues. *That* is the
origin of the quantum false positives.

In [6]:
fig, ax = plt.subplots(figsize=(10, 5.5))
lg = np.linspace(1.0, 7.0, 500)
ax.plot(lg, f_filter(lg), color="#555555", lw=2, label=r"1-bit filter $f(\lambda)=\cos(\lambda t/2)$")
ax.axhline(0, color="k", lw=0.6); ax.axvline(SDIAG, color="#2166ac", ls="--", lw=1.5, label=f"notch $\\lambda=\\gamma+\\delta={SDIAG:.0f}$")
ax.scatter(eA, f_filter(eA), s=160, marker="X", color="#d6604d", ec="k", zorder=6,
           label="(A) isolated false  -> erased")
ax.scatter(eB, f_filter(eB), s=110, color="#1b7837", ec="k", zorder=6,
           label="(B) real track  -> kept")
ax.scatter(eC, f_filter(eC), s=110, marker="s", color="#e08214", ec="k", zorder=6,
           label="(C) bridge (false)  -> kept")
ax.set_xlabel(r"eigenvalue $\lambda$"); ax.set_ylabel(r"amplitude kept, $f(\lambda)$")
ax.set_title("Good vs bad eigenvalues: on the notch = erased, off the notch = kept", fontweight="bold")
ax.legend(fontsize=9, loc="lower left"); ax.grid(alpha=0.3)
fig.tight_layout()
for e, dp in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"filter_with_examples.{e}", dpi=dp, bbox_inches="tight", facecolor="white")
plt.show(); print("saved filter_with_examples")

saved filter_with_examples


## 3b. Explicit low-dimensional matrices — and *more than one* bad eigenvalue
Each cluster's block is small enough to write by hand. Because $`A=(\gamma+\delta)I-C`$
(with $`C`$ the 0/1 compatibility adjacency), an eigenvalue is **bad** (on the notch,
erased by the quantum filter) **exactly when $`C`$ has a zero adjacency eigenvalue**:
$`\mu(C)=0\Rightarrow\lambda=\gamma+\delta`$. We solve $`A\mathbf{x}=\delta\mathbf{1}`$ by hand
for four shapes (γ=3, δ=1, so $`s\equiv\gamma+\delta=4`$, notch at $`\lambda=4`$).

In [7]:
import numpy as np
s_, dlt = 4.0, 1.0; t_ = np.pi/s_; tau_ = 0.35; ff = lambda L: np.cos(L*t_/2)
def worked(name, A):
    A = np.array(A, float); n = A.shape[0]; b = np.ones(n)
    w, U = np.linalg.eigh(A); beta = U.T @ b; xC = np.linalg.solve(A, b)
    xqr = U @ (beta*ff(w)); m = xC > tau_
    xQ = np.abs(xqr)*(np.linalg.norm(xC[m])/np.linalg.norm(xqr[m])) if m.any() and np.linalg.norm(xqr[m])>0 else np.abs(xqr)
    nbad = int((np.abs(w-s_) < 1e-9).sum())
    print(f"{name}:  eigenvalues={np.round(w,3)}   BAD(on notch)={nbad}")
    print(f"    classical x={np.round(xC,3)}   quantum x={np.round(xQ,3)}   (active if >{tau_})")
    return w, nbad
EX = {
  "clean track  P4  ": [[4,-1,0,0],[-1,4,-1,0],[0,-1,4,-1],[0,0,-1,4]],
  "isolated     1x1 ": [[4]],
  "bridge       P3  ": [[4,-1,0],[-1,4,-1],[0,-1,4]],
  "hub  K(1,3)  star": [[4,-1,-1,-1],[-1,4,0,0],[-1,0,4,0],[-1,0,0,4]],
  "hub  K(1,4)  star": [[4,-1,-1,-1,-1],[-1,4,0,0,0],[-1,0,4,0,0],[-1,0,0,4,0],[-1,0,0,0,4]],
}
specs = {k: worked(k, v) for k, v in EX.items()}
print("\nGeneral: star K(1,m) has C-spectrum {sqrt(m), 0 x(m-1), -sqrt(m)} -> A has (m-1) eigenvalues AT the notch.")

# ladder figure: each example's eigenvalues vs lambda, bad ones ringed; filter on a twin axis
fig, ax = plt.subplots(figsize=(11, 5.5))
lg = np.linspace(1.2, 6.8, 400); axf = ax.twinx()
axf.plot(lg, ff(lg), color="0.7", lw=1.6, zorder=0); axf.axhline(0, color="0.7", lw=0.6)
axf.set_ylabel("filter $f(\\lambda)=\\cos(\\lambda t/2)$", color="0.5"); axf.set_ylim(-1.1, 1.1)
rows = list(specs.keys())
for r, name in enumerate(rows):
    w, nbad = specs[name]
    wj = w.astype(float).copy()
    for v in np.unique(np.round(wj, 6)):                 # jitter exact duplicates so they're visible
        idx = np.where(np.abs(wj-v) < 1e-6)[0]
        if len(idx) > 1: wj[idx] += np.linspace(-0.06, 0.06, len(idx))
    bad = np.abs(w-s_) < 1e-9
    ax.scatter(wj[~bad], [r]*int((~bad).sum()), s=80, color="#1b7837", ec="k", zorder=4, label="good" if r==0 else None)
    ax.scatter(wj[bad], [r]*int(bad.sum()), s=130, color="#d6604d", ec="k", marker="X", zorder=5, label="bad (notch)" if r==0 else None)
    ax.text(6.9, r, f"{nbad} bad", va="center", fontsize=9, fontweight="bold")
ax.axvline(s_, color="#2166ac", ls="--", lw=1.6, label="notch $\\lambda=\\gamma+\\delta=4$")
ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows, family="monospace", fontsize=9)
ax.set_xlabel("eigenvalue $\\lambda$"); ax.set_xlim(1.2, 7.6)
ax.set_title("Bad eigenvalues sit on the notch — a hub K(1,m) has $m-1$ of them", fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
for e, dp in (("pdf", 600), ("png", 300)):
    fig.savefig(_OUT / f"bad_eigenvalue_ladder.{e}", dpi=dp, bbox_inches="tight", facecolor="white")
plt.show(); print("saved bad_eigenvalue_ladder")

clean track  P4  :  eigenvalues=[2.382 3.382 4.618 5.618]   BAD(on notch)=0
    classical x=[0.364 0.455 0.455 0.364]   quantum x=[0.258 0.522 0.522 0.258]   (active if >0.35)
isolated     1x1 :  eigenvalues=[4.]   BAD(on notch)=1
    classical x=[0.25]   quantum x=[0.]   (active if >0.35)
bridge       P3  :  eigenvalues=[2.586 4.    5.414]   BAD(on notch)=1
    classical x=[0.357 0.429 0.357]   quantum x=[0.27  0.541 0.27 ]   (active if >0.35)
hub  K(1,3)  star:  eigenvalues=[2.268 4.    4.    5.732]   BAD(on notch)=2
    classical x=[0.538 0.385 0.385 0.385]   quantum x=[0.742 0.247 0.247 0.247]   (active if >0.35)
hub  K(1,4)  star:  eigenvalues=[2. 4. 4. 4. 6.]   BAD(on notch)=3
    classical x=[0.667 0.417 0.417 0.417 0.417]   quantum x=[0.955 0.239 0.239 0.239 0.239]   (active if >0.35)

General: star K(1,m) has C-spectrum {sqrt(m), 0 x(m-1), -sqrt(m)} -> A has (m-1) eigenvalues AT the notch.


saved bad_eigenvalue_ladder


**Reading it off.** A *clean track* (path $`P_4`$) has $`C`$-eigenvalues
$`2\cos\frac{k\pi}{5}\ne0`$ — **no** bad eigenvalue, so it survives. An *isolated false*
segment is the $`1\times1`$ block $`[\gamma+\delta]`$ — its single eigenvalue **is** the notch
(1 bad), and since $`\mathbf{b}`$ excites it, the quantum filter sends it to $`0`$ (correctly
removed). A *bridge* ($`P_3`$) has $`C`$-eigenvalues $`\{\sqrt2,0,-\sqrt2\}`$ — **one** bad
eigenvalue at the notch. A *hub* where one segment is compatible with $`m`$ others
(star $`K_{1,m}`$) has $`C`$-eigenvalues $`\{\sqrt m,\,0^{(m-1)},\,-\sqrt m\}`$ — **$`m-1`$ bad
eigenvalues**: $`K_{1,3}`$ gives 2, $`K_{1,4}`$ gives 3. So *more than one* bad eigenvalue
is the rule in dense hubs, found analytically. **Crucially**, in the coupled cases
the symmetric source $`\mathbf{b}=\delta\mathbf{1}`$ does **not** excite those (antisymmetric)
notch eigenvectors ($`\beta=0`$), so the activation is carried entirely by the
*off-notch* modes — which the filter keeps. That is why the bridge/hub **false**
segments survive ($`x_Q>\tau`$) while the isolated false segment is annihilated.

## 4. So, in plain words

- The detector sees **hits**; we draw every possible **segment** (a link to the next
  plane). A real **track** is 4 true segments in a row; everything else is a false
  segment, and there are $`\sim n_{\rm seg}=4T^2`$ of them — the haystack.
- The matrix $`A`$ only couples segments that *could* be the same particle. So the
  picture splits into tiny independent **clusters**. **99.5%** of segments are
  **isolated** (no compatible partner) — almost all of them false.
- Each isolated segment is its own eigenvector with eigenvalue **exactly $`\gamma+\delta`$**
  — the **bad** value, because the 1-bit filter is **zero** there. The quantum
  solver therefore **erases the entire isolated bulk** (amplitude $`\sim10^{-14}`$),
  which is *exactly right*: those are false. (Classically they sit at $`0.25`$, just
  below threshold; the quantum solver is even cleaner here.)
- A **real track** is a 4-segment cluster whose eigenvalues are **off the notch**
  ("good"), so the filter keeps it — the track is reconstructed (it keeps 3/4
  segments; the lone outer one is pushed to $`0.18`$, the "plateau halving", which is
  why per-track efficiency is $`\approx75\%`$).
- A **cross-track bridge** is an *accidental* cluster of false segments — two
  different tracks' hits that happen to line up and share a hit. It **also** has
  off-notch ("good") eigenvalues, so the filter keeps it too — and it becomes a
  **false positive**. Every surviving false segment connects **two different
  tracks**.

**The crux:** "good" vs "bad" eigenvalue just means *off* vs *on* the notch. The
notch cleanly kills the isolated false bulk (the easy 99.5%). The leftover false
positives are the rare false segments that have organised into a coupled cluster —
they carry good eigenvalues and are indistinguishable, at the eigenvalue level,
from a genuine track.

## 5. Classical vs quantum — the explicit comparison
Same matrix, two inversions: classical $`1/\lambda`$ vs the 1-bit $`f(\lambda)=\cos(\lambda t/2)`$.
**Analytic results:** the quantum filter *amplifies* a hub core (centre/leaf ratio
$`=m`$ vs classical $`(s{+}m)/(s{+}1)`$) and *suppresses* a true track's outer segments
($`4/11\to`$ halved); together these make the quantum true/false activations overlap,
so at matched efficiency the quantum false rate explodes.

In [8]:
sC = 4.0; tt = np.pi/sC; tauv = 0.35; ff = lambda L: np.cos(L*tt/2)
def cl(A):
    A = np.array(A, float); b = np.ones(A.shape[0]); w, U = np.linalg.eigh(A); be = U.T@b
    xC = np.linalg.solve(A, b); xq = U@(be*ff(w)); m = xC > tauv
    xQ = np.abs(xq)*(np.linalg.norm(xC[m])/np.linalg.norm(xq[m])) if m.any() and np.linalg.norm(xq[m])>0 else np.abs(xq)
    return xC, xQ
P1=cl([[4]]); P2=cl([[4,-1],[-1,4]]); P3=cl([[4,-1,0],[-1,4,-1],[0,-1,4]])
P4=cl([[4,-1,0,0],[-1,4,-1,0],[0,-1,4,-1],[0,0,-1,4]])
H3=cl([[4,-1,-1,-1],[-1,4,0,0],[-1,0,4,0],[-1,0,0,4]])
labs = ["isolated\n(false)","pair\n(false)","bridge mid\n(false)","track outer\n(TRUE)","track inner\n(TRUE)","hub centre\n(false)","hub leaf\n(false)"]
xcv = [P1[0][0],P2[0][0],P3[0][1],P4[0][0],P4[0][1],H3[0][0],H3[0][1]]
xqv = [P1[1][0],P2[1][0],P3[1][1],P4[1][0],P4[1][1],H3[1][0],H3[1][1]]
istrue = np.array([0,0,0,1,1,0,0], bool)
fig, ax = plt.subplots(1, 3, figsize=(17, 5)); xpos = np.arange(len(labs)); ww = 0.4
for i in np.where(istrue)[0]: ax[0].axvspan(i-0.5, i+0.5, color="green", alpha=0.07)
ax[0].bar(xpos-ww/2, xcv, ww, color="#1f78b4", label="classical $1/\\lambda$")
ax[0].bar(xpos+ww/2, xqv, ww, color="#e31a1c", label="quantum $\\cos(\\lambda t/2)$")
ax[0].axhline(tauv, color="k", ls="--", lw=1.3, label="$\\tau=0.35$")
ax[0].set_xticks(xpos); ax[0].set_xticklabels(labs, fontsize=7.5); ax[0].set_ylabel("activation")
ax[0].set_title("(a) Q erases isolated, suppresses true-outer, amplifies hub-centre", fontweight="bold", fontsize=9)
ax[0].legend(fontsize=8)
mm = np.arange(2, 7)
ax[1].plot(mm, (sC+mm)/(sC+1), "o-", color="#1f78b4", lw=2, label="classical $(s{+}m)/(s{+}1)$")
ax[1].plot(mm, mm, "s-", color="#e31a1c", lw=2, label="quantum $m$")
ax[1].set_xlabel("hub degree $m$"); ax[1].set_ylabel("centre / leaf ratio")
ax[1].set_title("(b) Quantum filter amplifies the hub core", fontweight="bold", fontsize=9); ax[1].legend(fontsize=9)
thr = np.unique(np.round(solQ, 4))
effq = np.array([(truth & (solQ > th)).sum()/truth.sum() for th in thr])
farq = np.array([(~truth & (solQ > th)).sum()/max((solQ > th).sum(), 1) for th in thr]); o = np.argsort(effq)
ax[2].plot(effq[o]*100, farq[o]*100, "-", color="#e31a1c", lw=2, label="quantum (sweep $\\tau$)")
aCl = solC > tauv
ax[2].scatter([(truth&aCl).sum()/truth.sum()*100], [(~truth&aCl).sum()/max(aCl.sum(),1)*100], s=130, color="#1f78b4", ec="k", zorder=5, label="classical @ $\\tau=0.35$")
aQl = solQ > tauv
ax[2].scatter([(truth&aQl).sum()/truth.sum()*100], [(~truth&aQl).sum()/max(aQl.sum(),1)*100], s=110, color="#e31a1c", ec="k", marker="D", zorder=5, label="quantum @ $\\tau=0.35$")
ax[2].set_xlabel("efficiency (%)"); ax[2].set_ylabel("false rate (%)")
ax[2].set_title("(c) Operating characteristic (T=400):\nmatched efficiency -> quantum far >> classical", fontweight="bold", fontsize=9); ax[2].legend(fontsize=8.5)
fig.tight_layout()
for e, dp in (("pdf", 600), ("png", 300)):
    fig.savefig(_OUT / f"classical_vs_quantum.{e}", dpi=dp, bbox_inches="tight", facecolor="white")
plt.show()
print("per-cluster classical x:", np.round(xcv,3)); print("per-cluster quantum   x:", np.round(xqv,3)); print("saved classical_vs_quantum")

per-cluster classical x: [0.25  0.333 0.429 0.364 0.455 0.538 0.385]
per-cluster quantum   x: [0.    0.383 0.541 0.258 0.522 0.742 0.247]
saved classical_vs_quantum
